# 🏆 CAN LLMs BEAT XGBOOST ?
## Hackathon — Forecasting des Courbes de Charge ENEDIS
**École des Ponts ParisTech | Charif EL JAZOULI | 20 Avril 2026**

---

### 🎯 Objectif
Prédire les **672 demi-heures** (14 jours) de la courbe de charge nationale ENEDIS
du **18/12/2024 00:00** au **31/12/2024 23:30**.

### 📋 Ce notebook est votre starter kit
- ✅ Chargement des données ENEDIS et météo
- ✅ Baseline Naive déjà codée
- ✅ Structure XGBoost à compléter
- ✅ Structure LLMs à compléter
- ✅ Validation et export CSV

**Votre équipe** : _à remplir_  
**Modèle soumis** : _XGBoost / Chronos / Moirai / TimesFM_

---

## 0. Installation & Imports
> Lancez cette cellule en premier. L'installation prend ~5 minutes.

In [ ]:
# ── Installation (à lancer une seule fois) ────────────────────────────────
import subprocess, sys

packages = [
    'pandas numpy matplotlib seaborn scikit-learn',
    'statsmodels xgboost',
    'openmeteo-requests requests-cache retry-requests',
    'chronos-forecasting',       # Amazon Chronos
    'uni2ts gluonts',            # Salesforce Moirai
    'timesfm',                   # Google TimesFM
    'torch',
]
for pkg in packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkg.split())
print('✅ Installation terminée')

In [ ]:
# ── Imports principaux ────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

# Config affichage
pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 11

# ── Constantes du hackathon ────────────────────────────────────────────────
TEAM_NAME    = 'MON_EQUIPE'   # ← CHANGEZ ICI
TRAIN_END    = '2024-12-17 23:30:00'
TEST_START   = '2024-12-18 00:00:00'
TEST_END     = '2024-12-31 23:30:00'
HORIZON      = 672   # 14 jours × 48 demi-heures

print(f'🏷️  Équipe : {TEAM_NAME}')
print(f'📅 Train  : jusqu\'au {TRAIN_END}')
print(f'📅 Test   : {TEST_START} → {TEST_END}  ({HORIZON} points)')

## 1. Chargement des données ENEDIS

**Source** : [data.enedis.fr](https://data.enedis.fr) → Courbes de charge générées

Téléchargez le dataset *'Données de courbes de charge'* en CSV et placez-le
dans le même dossier que ce notebook, ou utilisez l'API directement ci-dessous.

> 💡 Le dataset couvre la puissance soutirée reconstituée nationale (MW) en pas 30 min.

In [ ]:
# ── Option A : Téléchargement direct via l'API ENEDIS ────────────────────
# (peut être lent selon la connexion — Option B si ça bloque)

import urllib.request
import io

def load_enedis_from_api():
    """
    Télécharge les courbes de charge agrégées nationales depuis data.enedis.fr
    Dataset : 'Données de courbes de charge générées'
    """
    # URL de l'API ENEDIS Open Data (dataset courbes de charge)
    url = (
        'https://data.enedis.fr/api/explore/v2.1/catalog/datasets/'
        'donnees-de-courbes-de-charge-agregees-nationales/exports/csv'
        '?lang=fr&timezone=Europe%2FParis&use_labels=true&delimiter=%3B'
    )
    print('⬇️  Téléchargement en cours...')
    response = urllib.request.urlopen(url, timeout=60)
    content  = response.read().decode('utf-8')
    df = pd.read_csv(io.StringIO(content), sep=';', parse_dates=True)
    print(f'✅ {len(df):,} lignes téléchargées')
    print(f'Colonnes disponibles : {df.columns.tolist()}')
    return df

# df_raw = load_enedis_from_api()  # ← décommentez pour tester

# ── Option B : Chargement depuis fichier local ─────────────────────────────
# Placez votre CSV téléchargé sur data.enedis.fr dans le même dossier
# df_raw = pd.read_csv('courbes_charge_enedis.csv', sep=';')

print('📌 Choisissez Option A (API) ou Option B (fichier local)')
print('   Puis adaptez le parsing ci-dessous selon les colonnes de votre fichier')

In [ ]:
# ── Parsing et nettoyage ─────────────────────────────────────────────────
# ADAPTEZ ces lignes selon les colonnes réelles de votre dataset ENEDIS
# Les noms de colonnes varient selon la version du dataset téléchargé

# df_raw.head(3)  # ← décommentez pour voir les colonnes

# Exemple de parsing (à adapter) :
# df = df_raw[['Horodate', 'Total réseau']].copy()
# df.columns = ['datetime', 'load_mw']
# df['datetime'] = pd.to_datetime(df['datetime'], utc=True).dt.tz_convert('Europe/Paris').dt.tz_localize(None)
# df = df.set_index('datetime').sort_index()
# df['load_mw'] = pd.to_numeric(df['load_mw'], errors='coerce')
# df = df.dropna()

# ── Vérification ─────────────────────────────────────────────────────────
# print(f'Période : {df.index.min()} → {df.index.max()}')
# print(f'Points  : {len(df):,}  |  Fréquence : {pd.infer_freq(df.index)}')
# print(f'Missing : {df.isna().sum().values[0]}')
# df.describe()

print('⬆️  Complétez le parsing ci-dessus avec vos colonnes réelles')

In [ ]:
# ── Resample pour garantir la fréquence 30T exacte ───────────────────────
# (utile si certains timestamps sont manquants ou irréguliers)

# df = df.resample('30T').interpolate(method='time')

# ── Split train / test ────────────────────────────────────────────────────
# df_train = df[:TRAIN_END]
# df_test  = df[TEST_START:TEST_END]   # CE QUE VOUS DEVEZ PRÉDIRE

# print(f'Train : {len(df_train):,} pts')
# print(f'Test  : {len(df_test):,} pts  ← à prédire (vous ne verrez les vrais que lors du scoring)')

# ── Visualisation rapide ──────────────────────────────────────────────────
# fig, ax = plt.subplots(figsize=(16, 4))
# df_train['load_mw'].resample('D').mean().plot(ax=ax, label='Train', color='steelblue')
# ax.axvline(pd.Timestamp(TEST_START), color='red', linestyle='--', label='Début test')
# ax.set_title('Courbe de charge ENEDIS — Vue d\'ensemble')
# ax.set_ylabel('Puissance (MW)')
# ax.legend(); ax.grid(alpha=0.3)
# plt.tight_layout()

print('✅ Structure prête — remplissez df_train')

## 2. Données météo — Open-Meteo

**Source** : [open-meteo.com](https://open-meteo.com) — Gratuit, sans clé API

Nous récupérons la température pour 4 grandes villes françaises
et calculons une moyenne pondérée représentant la France entière.

In [ ]:
import openmeteo_requests
import requests_cache
from retry_requests import retry

def fetch_meteo_france(start_date: str, end_date: str) -> pd.DataFrame:
    """
    Récupère la température horaire pour 4 villes françaises
    et retourne une moyenne pondérée au pas 30 min.
    """
    # Cache pour éviter de re-télécharger
    session = requests_cache.CachedSession('.meteo_cache', expire_after=3600)
    session = retry(session, retries=3, backoff_factor=0.2)
    om = openmeteo_requests.Client(session=session)

    # 4 villes + poids (population relative)
    villes = [
        {'name': 'Paris',     'lat': 48.8566, 'lon': 2.3522,  'weight': 0.35},
        {'name': 'Lyon',      'lat': 45.7640, 'lon': 4.8357,  'weight': 0.20},
        {'name': 'Marseille', 'lat': 43.2965, 'lon': 5.3698,  'weight': 0.20},
        {'name': 'Lille',     'lat': 50.6292, 'lon': 3.0573,  'weight': 0.25},
    ]

    dfs = []
    for v in villes:
        params = {
            'latitude': v['lat'], 'longitude': v['lon'],
            'hourly': ['temperature_2m', 'shortwave_radiation', 'wind_speed_10m'],
            'start_date': start_date, 'end_date': end_date,
            'timezone': 'Europe/Paris'
        }
        resp  = om.weather_api('https://archive-api.open-meteo.com/v1/archive', params=params)
        hourly = resp[0].Hourly()
        idx   = pd.date_range(
            start=pd.Timestamp(hourly.Time(), unit='s', tz='Europe/Paris').tz_localize(None),
            periods=hourly.VariablesLength() and len(hourly.Variables(0).ValuesAsNumpy()),
            freq='H'
        )
        df_v = pd.DataFrame({
            'temp_c':       hourly.Variables(0).ValuesAsNumpy() * v['weight'],
            'radiation':    hourly.Variables(1).ValuesAsNumpy() * v['weight'],
            'wind_kmh':     hourly.Variables(2).ValuesAsNumpy() * v['weight'],
        }, index=idx)
        dfs.append(df_v)

    df_meteo = sum(dfs)   # somme pondérée
    # Resample 30 min par interpolation
    df_meteo = df_meteo.resample('30T').interpolate(method='time')
    print(f'✅ Météo chargée : {len(df_meteo):,} pts  ({start_date} → {end_date})')
    return df_meteo

# Chargement (peut prendre 1-2 min la première fois, puis mis en cache)
df_meteo = fetch_meteo_france('2021-01-01', '2024-12-31')
df_meteo.tail(3)

In [ ]:
# ── Merge avec les données ENEDIS ─────────────────────────────────────────
# df = df.join(df_meteo, how='left')
# df[['temp_c', 'radiation', 'wind_kmh']] = df[['temp_c', 'radiation', 'wind_kmh']].interpolate()

# df_train = df[:TRAIN_END]
# df_test  = df[TEST_START:TEST_END]

# print('Aperçu du dataset enrichi :')
# df_train.tail(3)

print('⬆️  Décommentez après avoir chargé df ENEDIS')

## 3. Exploration rapide (EDA)

> 10 minutes max — l'objectif est de comprendre vos données, pas de les analyser exhaustivement.

In [ ]:
# ── Profil journalier et hebdomadaire ────────────────────────────────────
# fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Profil intra-journalier
# df_train.groupby(df_train.index.hour)['load_mw'].mean().plot(
#     ax=axes[0], color='steelblue', marker='o', markersize=4)
# axes[0].set_title('Profil moyen par heure')
# axes[0].set_xlabel('Heure'); axes[0].grid(alpha=0.3)

# Profil hebdomadaire
# days = ['Lun', 'Mar', 'Mer', 'Jeu', 'Ven', 'Sam', 'Dim']
# df_train.groupby(df_train.index.dayofweek)['load_mw'].mean().plot(
#     ax=axes[1], kind='bar', color='coral', edgecolor='black')
# axes[1].set_xticklabels(days, rotation=0)
# axes[1].set_title('Profil moyen par jour de semaine')
# axes[1].grid(axis='y', alpha=0.3)

# plt.suptitle('Saisonnalités de la consommation ENEDIS', fontsize=13, fontweight='bold')
# plt.tight_layout()

print('💡 Décommentez pour visualiser')

In [ ]:
# ── Corrélation température / charge ─────────────────────────────────────
# fig, ax = plt.subplots(figsize=(8, 6))
# sample = df_train.resample('H').mean().dropna()
# sc = ax.scatter(sample['temp_c'], sample['load_mw'],
#                 c=sample.index.month, cmap='RdYlBu_r',
#                 alpha=0.15, s=3)
# plt.colorbar(sc, ax=ax, label='Mois')
# ax.set_xlabel('Température (°C)')
# ax.set_ylabel('Charge (MW)')
# ax.set_title('Relation Température / Consommation par mois')
# ax.grid(alpha=0.3)
# plt.tight_layout()

print('💡 Décommentez pour visualiser')

## 4. Métriques d'évaluation

> Ces fonctions sont fixes — ne les modifiez pas.

In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray,
                     y_train: np.ndarray = None,
                     model_name: str = '') -> dict:
    """
    Calcule MAE, RMSE, MAPE et MASE.
    MAE est la métrique officielle du hackathon.
    """
    y_true = np.array(y_true).flatten()
    y_pred = np.array(y_pred).flatten()[:len(y_true)]

    mae  = float(np.mean(np.abs(y_true - y_pred)))
    rmse = float(np.sqrt(np.mean((y_true - y_pred)**2)))
    mape = float(np.mean(np.abs((y_true - y_pred) / (np.abs(y_true) + 1e-8))) * 100)

    if y_train is not None:
        naive_errors = np.abs(np.diff(y_train, n=48))
        naive_mae    = float(np.mean(naive_errors))
        mase = mae / (naive_mae + 1e-8)
    else:
        mase = None

    result = {
        'Modèle':    model_name,
        'MAE (MW)':  round(mae, 1),
        'RMSE (MW)': round(rmse, 1),
        'MAPE (%)':  round(mape, 2),
        'MASE':      round(mase, 4) if mase else '-'
    }
    return result

all_results = []   # stockage de tous les résultats
print('✅ Métriques prêtes')

## 5. Baseline — Naive Saisonnière

La baseline la plus simple. Elle est **déjà codée** — faites-la tourner pour voir
le score à battre minimum.

In [ ]:
class NaiveSeasonal:
    """
    Baseline : y_hat(t) = y(t - season_length)
    Utilise le dernier cycle connu pour prédire les H prochains pas.
    """
    def __init__(self, season_length: int = 48):
        self.s = season_length
        self.tail_ = None

    def fit(self, y: np.ndarray):
        # Garde les 2 dernières semaines pour couvrir n'importe quel horizon
        self.tail_ = y[-(self.s * 2):]
        return self

    def predict(self, h: int) -> np.ndarray:
        reps = int(np.ceil(h / self.s))
        return np.tile(self.tail_[-self.s:], reps)[:h]


# ── Naive journalière (lag 48 = J-1) ─────────────────────────────────────
# naive_j = NaiveSeasonal(season_length=48)
# naive_j.fit(df_train['load_mw'].values)
# y_pred_naive_j = naive_j.predict(HORIZON)

# ── Naive hebdomadaire (lag 48*7 = S-1) ──────────────────────────────────
# naive_w = NaiveSeasonal(season_length=48 * 7)
# naive_w.fit(df_train['load_mw'].values)
# y_pred_naive_w = naive_w.predict(HORIZON)

# ── Évaluation (sur les derniers 672 points du train comme proxy) ─────────
# y_proxy = df_train['load_mw'].values[-HORIZON:]
# y_train_arr = df_train['load_mw'].values

# r_naive_j = compute_metrics(y_proxy,
#     NaiveSeasonal(48).fit(df_train['load_mw'].values[:-HORIZON]).predict(HORIZON),
#     y_train_arr, 'Naive Journalier')
# r_naive_w = compute_metrics(y_proxy,
#     NaiveSeasonal(48*7).fit(df_train['load_mw'].values[:-HORIZON]).predict(HORIZON),
#     y_train_arr, 'Naive Hebdomadaire')
# all_results += [r_naive_j, r_naive_w]
# pd.DataFrame([r_naive_j, r_naive_w])

print('⬆️  Décommentez pour faire tourner la baseline')

## 6. 🌲 XGBoost — Feature Engineering

C'est ici que votre créativité fait la différence.
Les features de base sont fournies — ajoutez les vôtres !

> **Rappel** : encodage cyclique obligatoire pour l'heure et le jour.

In [ ]:
import xgboost as xgb

# ── Feature engineering temporel ─────────────────────────────────────────
def create_features(df: pd.DataFrame, target: str = 'load_mw') -> pd.DataFrame:
    df = df.copy()

    # --- Features calendaires brutes ---
    df['hour']       = df.index.hour
    df['dayofweek']  = df.index.dayofweek
    df['month']      = df.index.month
    df['dayofyear']  = df.index.dayofyear
    df['is_weekend'] = (df.index.dayofweek >= 5).astype(int)

    # --- Encodage cyclique (NE PAS utiliser hour/dayofweek bruts) ---
    df['sin_30min']  = np.sin(2 * np.pi * (df.index.hour * 2 + df.index.minute // 30) / 48)
    df['cos_30min']  = np.cos(2 * np.pi * (df.index.hour * 2 + df.index.minute // 30) / 48)
    df['sin_dow']    = np.sin(2 * np.pi * df.index.dayofweek / 7)
    df['cos_dow']    = np.cos(2 * np.pi * df.index.dayofweek / 7)
    df['sin_month']  = np.sin(2 * np.pi * df.index.month / 12)
    df['cos_month']  = np.cos(2 * np.pi * df.index.month / 12)

    # --- Jours fériés français (décembre clé pour le test !) ---
    jours_feries = [
        '2021-12-25', '2022-12-25', '2023-12-25', '2024-12-25',  # Noël
        '2021-01-01', '2022-01-01', '2023-01-01', '2024-01-01',  # Jour de l'An
        # Ajoutez tous les jours fériés ici !
    ]
    holidays = pd.to_datetime(jours_feries)
    df['is_holiday'] = df.index.normalize().isin(holidays).astype(int)

    # --- Lags (attention au data leakage : lag minimum = HORIZON) ---
    for lag in [48, 48*2, 48*7, 48*14]:
        df[f'lag_{lag}'] = df[target].shift(lag)

    # --- Rolling features ---
    for w in [48, 48*7]:
        df[f'roll_mean_{w}'] = df[target].shift(48).rolling(w).mean()
        df[f'roll_std_{w}']  = df[target].shift(48).rolling(w).std()

    # ══════════════════════════════════════════════════════════════
    # ✏️  ZONE LIBRE — Ajoutez vos propres features ici !
    # Idées : température, interaction temp*heure, temp², ...
    # ══════════════════════════════════════════════════════════════

    return df

print('✅ Fonction create_features définie')

In [ ]:
# ── Construction du dataset ───────────────────────────────────────────────
# df_feat = create_features(df)
# df_feat = df_feat.dropna()

# FEATURE_COLS = [c for c in df_feat.columns if c not in ['load_mw']]
# print(f'{len(FEATURE_COLS)} features : {FEATURE_COLS}')

# X_train = df_feat.loc[:TRAIN_END][FEATURE_COLS]
# y_train = df_feat.loc[:TRAIN_END]['load_mw']

# # Pour prédire le test, on utilise les features temporelles + lags depuis le train
# # (les lags pointent vers des dates dans le train — pas de leakage)
# X_test_feat = df_feat.loc[TEST_START:TEST_END][FEATURE_COLS]

print('⬆️  Décommentez après avoir chargé df')

In [ ]:
# ── Entraînement XGBoost ──────────────────────────────────────────────────
# xgb_model = xgb.XGBRegressor(
#     n_estimators     = 1000,
#     learning_rate    = 0.05,
#     max_depth        = 6,
#     subsample        = 0.8,
#     colsample_bytree = 0.8,
#     min_child_weight = 3,
#     reg_alpha        = 0.01,
#     reg_lambda       = 1.0,
#     tree_method      = 'hist',
#     early_stopping_rounds = 50,
#     random_state     = 42,
#     n_jobs           = -1,
# )

# # Validation sur les 2 dernières semaines du train
# val_size = HORIZON
# X_val = X_train.iloc[-val_size:]
# y_val = y_train.iloc[-val_size:]
# X_tr  = X_train.iloc[:-val_size]
# y_tr  = y_train.iloc[:-val_size]

# xgb_model.fit(X_tr, y_tr,
#               eval_set=[(X_val, y_val)],
#               verbose=100)

# y_pred_xgb_val = xgb_model.predict(X_val)
# r_xgb = compute_metrics(y_val.values, y_pred_xgb_val, y_tr.values, 'XGBoost')
# all_results.append(r_xgb)
# print('XGBoost validation :', r_xgb)

# # Prédiction finale sur le test
# y_pred_xgb_test = xgb_model.predict(X_test_feat)

print('⬆️  Décommentez pour entraîner XGBoost')

In [ ]:
# ── Feature importance ────────────────────────────────────────────────────
# feat_imp = pd.Series(xgb_model.feature_importances_, index=FEATURE_COLS)
# feat_imp.nlargest(15).plot(kind='barh', figsize=(10, 6), color='steelblue')
# plt.title('Top 15 Features — XGBoost')
# plt.xlabel('Importance')
# plt.tight_layout()

print('💡 Décommentez pour visualiser les features importantes')

## 7. ⚡ LLM — Amazon Chronos

Modèle de fondation basé sur T5. Traite les valeurs numériques comme des tokens.
Prévision **zero-shot** : aucun réentraînement nécessaire.

> 💡 Testez différentes longueurs de contexte : 336 (1 sem), 672 (2 sem), 1344 (1 mois)

In [ ]:
import torch
from chronos import ChronosPipeline

# ── Chargement du modèle ──────────────────────────────────────────────────
# Tailles disponibles : tiny, mini, small, base, large
# 'small' est un bon compromis vitesse/précision pour un hackathon

# print('⬇️  Chargement de Chronos (1ère fois : ~2 min)...')
# chronos_pipe = ChronosPipeline.from_pretrained(
#     'amazon/chronos-t5-small',
#     device_map='auto',
#     torch_dtype=torch.bfloat16,
# )
# print('✅ Chronos chargé')

print('⬆️  Décommentez pour charger Chronos')

In [ ]:
# ── Prévision Chronos ─────────────────────────────────────────────────────
# CONTEXT_LEN = 672   # ← testez 336, 672, 1344

# context = torch.tensor(
#     df_train['load_mw'].values[-CONTEXT_LEN:],
#     dtype=torch.float32
# )

# forecast = chronos_pipe.predict(
#     context=context.unsqueeze(0),   # [1, context_len]
#     prediction_length=HORIZON,
#     num_samples=100,
#     temperature=1.0,
#     top_k=50,
# )
# # forecast : [1, 100, HORIZON]

# y_pred_chronos  = np.median(forecast[0].numpy(), axis=0)  # médiane
# y_chronos_q10   = np.percentile(forecast[0].numpy(), 10, axis=0)
# y_chronos_q90   = np.percentile(forecast[0].numpy(), 90, axis=0)

# # Évaluation proxy (sur derniers 672 pts du train)
# context_eval = torch.tensor(
#     df_train['load_mw'].values[-(CONTEXT_LEN + HORIZON):-HORIZON],
#     dtype=torch.float32)
# forecast_eval = chronos_pipe.predict(context_eval.unsqueeze(0), HORIZON, 100)
# y_eval = df_train['load_mw'].values[-HORIZON:]
# r_chronos = compute_metrics(y_eval,
#     np.median(forecast_eval[0].numpy(), axis=0),
#     df_train['load_mw'].values, 'Chronos')
# all_results.append(r_chronos)
# print('Chronos :', r_chronos)

print('⬆️  Décommentez pour faire tourner Chronos')

## 8. ⚡ LLM — Salesforce Moirai

Transformer universel entraîné sur LOTSA (27 milliards de points).
Supporte nativement les **covariables** — vous pouvez lui passer la température !

In [ ]:
# from uni2ts.model.moirai import MoiraiForecast, MoiraiModule
# from gluonts.dataset.pandas import PandasDataset
# from gluonts.dataset.split import split

# ── Préparation données au format GluonTS ─────────────────────────────────
# ds = PandasDataset(
#     dict(target=df['load_mw']),
#     freq='30T'
# )
# # Avec covariable température :
# # ds = PandasDataset(
# #     dict(target=df['load_mw'],
# #          past_feat_dynamic_real={'temp': df['temp_c']}),
# #     freq='30T')

# ── Chargement Moirai ─────────────────────────────────────────────────────
# moirai = MoiraiForecast(
#     module=MoiraiModule.from_pretrained('Salesforce/moirai-1.1-R-small'),
#     prediction_length=HORIZON,
#     context_length=672,
#     patch_size='auto',
#     num_samples=100,
#     target_dim=1,
# )

# train_ds, test_template = split(ds, offset=-HORIZON)
# test_ds = test_template.generate_instances(prediction_length=HORIZON, windows=1)
# predictor = moirai.create_predictor(batch_size=8)
# forecasts_moirai = list(predictor.predict(test_ds.input))

# y_pred_moirai = forecasts_moirai[0].mean
# r_moirai = compute_metrics(
#     df_test['load_mw'].values[:HORIZON],
#     y_pred_moirai,
#     df_train['load_mw'].values, 'Moirai')
# all_results.append(r_moirai)
# print('Moirai :', r_moirai)

print('⬆️  Décommentez pour faire tourner Moirai')

## 9. ⚡ LLM — Google TimesFM

Decoder-only Transformer (200M params) de Google DeepMind.
Très rapide en inférence, excellent sur les séries haute fréquence.

In [ ]:
# import timesfm

# tfm = timesfm.TimesFm(
#     hparams=timesfm.TimesFmHparams(
#         backend='cpu',
#         per_core_batch_size=32,
#         horizon_len=HORIZON,
#     ),
#     checkpoint=timesfm.TimesFmCheckpoint(
#         huggingface_repo_id='google/timesfm-1.0-200m-pytorch'),
# )

# context_tfm = df_train['load_mw'].values[-672:].tolist()
# point_forecast, quantile_forecast = tfm.forecast(
#     [context_tfm],
#     freq=[0],  # 0 = haute fréquence
# )
# y_pred_timesfm = point_forecast[0]

# r_timesfm = compute_metrics(
#     df_test['load_mw'].values[:HORIZON],
#     y_pred_timesfm,
#     df_train['load_mw'].values, 'TimesFM')
# all_results.append(r_timesfm)
# print('TimesFM :', r_timesfm)

print('⬆️  Décommentez pour faire tourner TimesFM')

## 10. Comparaison des modèles

In [ ]:
# ── Tableau récapitulatif ─────────────────────────────────────────────────
# df_results = pd.DataFrame(all_results).set_index('Modèle')
# df_results = df_results.sort_values('MAE (MW)')
# print('\n🏆 CLASSEMENT (validation interne) :')
# print(df_results.to_string())

# ── Visualisation ─────────────────────────────────────────────────────────
# fig, axes = plt.subplots(1, 2, figsize=(16, 5))
# df_results['MAE (MW)'].plot(kind='barh', ax=axes[0], color='steelblue', edgecolor='black')
# axes[0].set_title('MAE par modèle (MW) — Plus bas = mieux')
# axes[0].axvline(df_results['MAE (MW)'].min(), color='red', linestyle='--', label='Best')
# axes[0].grid(axis='x', alpha=0.3)
# axes[0].legend()

# df_results['MASE'].plot(kind='barh', ax=axes[1], color='coral', edgecolor='black')
# axes[1].axvline(1.0, color='black', linestyle='--', label='Naive = 1.0')
# axes[1].set_title('MASE par modèle — Moins de 1 = bat la naive')
# axes[1].grid(axis='x', alpha=0.3)
# axes[1].legend()

# plt.suptitle(f'Hackathon — Équipe {TEAM_NAME}', fontsize=13, fontweight='bold')
# plt.tight_layout()

print('⬆️  Décommentez pour visualiser')

## 11. 📤 Export de la soumission

> **IMPORTANT** : Ne modifiez pas le code de validation.
Vérifiez que votre CSV est valide avant d'envoyer !

In [ ]:
def export_submission(y_pred: np.ndarray, team_name: str, model_name: str) -> str:
    """
    Génère et valide le CSV de soumission.
    Retourne le chemin du fichier créé.
    """
    y_pred = np.array(y_pred).flatten()

    # ── Validation ────────────────────────────────────────────────────────
    assert len(y_pred) == HORIZON, \
        f'❌ Attendu {HORIZON} prédictions, reçu {len(y_pred)}'

    assert not np.any(np.isnan(y_pred)), \
        '❌ Des valeurs NaN détectées dans vos prédictions'

    assert np.all(y_pred > 0), \
        '❌ Des valeurs négatives détectées (la puissance soutirée est toujours positive)'

    assert 20000 < np.mean(y_pred) < 100000, \
        f'❌ Moyenne anormale : {np.mean(y_pred):.0f} MW (attendu 30k-80k MW)'

    # ── Construction du DataFrame ─────────────────────────────────────────
    dates = pd.date_range(start=TEST_START, periods=HORIZON, freq='30T')
    df_sub = pd.DataFrame({
        'datetime':      dates.strftime('%Y-%m-%d %H:%M:%S'),
        'load_mw_pred':  np.round(y_pred, 2)
    })

    # ── Export ────────────────────────────────────────────────────────────
    filename = f'equipe_{team_name}_{model_name}_predictions.csv'
    df_sub.to_csv(filename, index=False)

    # ── Résumé ────────────────────────────────────────────────────────────
    print(f'✅  Fichier créé : {filename}')
    print(f'   Lignes       : {len(df_sub)} (attendu {HORIZON})')
    print(f'   Période      : {df_sub.datetime.iloc[0]} → {df_sub.datetime.iloc[-1]}')
    print(f'   Moy prédite  : {np.mean(y_pred):,.0f} MW')
    print(f'   Min / Max    : {np.min(y_pred):,.0f} / {np.max(y_pred):,.0f} MW')
    print(f'\n📧  Envoyez ce fichier à Charif EL JAZOULI')
    return filename

print('✅ Fonction export_submission prête')

In [ ]:
# ── Choisissez votre meilleur modèle et exportez ──────────────────────────

# Exemple avec XGBoost :
# export_submission(y_pred_xgb_test, TEAM_NAME, 'XGBoost')

# Exemple avec Chronos :
# export_submission(y_pred_chronos, TEAM_NAME, 'Chronos')

# Exemple avec TimesFM :
# export_submission(y_pred_timesfm, TEAM_NAME, 'TimesFM')

print('⬆️  Décommentez avec votre meilleur modèle !')

---
## 🏁 Bonne chance !

**Rappel des règles** :
- 1 soumission par équipe
- Fichier : `equipe_NOM_predictions.csv`
- 672 lignes, période 18/12/2024 → 31/12/2024
- Métrique officielle : **MAE (MW)**

*École des Ponts ParisTech | Charif EL JAZOULI | 20 Avril 2026*